# CESM2 Greenland Blocking Index

In [1]:
import os
import sys
import yaml
import copy

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
sys.path.insert(0, os.path.realpath('../libs/'))
import graph_utils as gu
#import verif_utils as vu

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [26]:
# Compute Greenland Blocking Index (GBI)

list_GBI = []

ilead = 0
# for ilead in range(10):
list_ds = []

for year in range(1958, 2020):
    # get data
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year}-11-01_daily_ensemble.zarr'
    ds = xr.open_zarr(fn)[['Z500']]
    ds = ds.mean('member')
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    # GBI definition
    Z500 = ds['Z500']
    Z500 = Z500.assign_coords(lon=((Z500.lon + 180) % 360) - 180).sortby("lon")
    
    # robust lat slicing
    if Z500.lat[0] < Z500.lat[-1]:
        Z500_GBI = Z500.sel(lat=slice(60, 80), lon=slice(-80, -20))
    else:
        Z500_GBI = Z500.sel(lat=slice(80, 60), lon=slice(-80, -20))
    
    Z500_mean = Z500.sel(lat=slice(60, 80))
    
    GBI = Z500_GBI.mean(('lat', 'lon')) - Z500_mean.mean(('lat', 'lon'))

    gp = GBI.groupby("time.year")
    GBI_mean  = gp.mean(dim="time",  skipna=True)
    GBI_30d = gp.map(lambda x: x.rolling(time=30, min_periods=30).mean().max(dim="time", skipna=True))

    ds_GBI = xr.Dataset(
        {"GBI_mean": GBI_mean, "GBI_30d": GBI_30d}
    )

    ds_GBI = ds_GBI.rename({'year': 'lead_year'})
    ds_GBI = ds_GBI.assign_coords({'lead_year': np.arange(10)})
    list_GBI.append(ds_GBI)

ds_GBI_all = xr.concat(list_GBI, dim='init_year')
ds_GBI_all = ds_GBI_all.assign_coords({'init_year': np.arange(1958, 2020)})

<xarray.Dataset>
Dimensions:    (lead_year: 10, init_year: 62)
Coordinates:
  * lead_year  (lead_year) int64 0 1 2 3 4 5 6 7 8 9
  * init_year  (init_year) int64 1958 1959 1960 1961 ... 2016 2017 2018 2019
Data variables:
    GBI_mean   (init_year, lead_year) float32 dask.array<chunksize=(1, 1), meta=np.ndarray>
    GBI_30d    (init_year, lead_year) float32 dask.array<chunksize=(1, 1), meta=np.ndarray>

In [28]:
save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/GBI.zarr'

ds_GBI_all = ds_GBI_all.chunk({'init_year': 62, 'lead_year': 10})
# ds_GBI_all.to_zarr(save_name)

In [29]:
xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/GBI.zarr')

<xarray.Dataset>
Dimensions:    (init_year: 62, lead_year: 10)
Coordinates:
  * init_year  (init_year) int64 1958 1959 1960 1961 ... 2016 2017 2018 2019
  * lead_year  (lead_year) int64 0 1 2 3 4 5 6 7 8 9
Data variables:
    GBI_30d    (init_year, lead_year) float32 dask.array<chunksize=(62, 10), meta=np.ndarray>
    GBI_mean   (init_year, lead_year) float32 dask.array<chunksize=(62, 10), meta=np.ndarray>